# Part 2: Tweet Sentiment Classification using Translation Encoder

This notebook demonstrates the transfer learning task from Part 2: reusing the pre-trained English-to-Chinese Transformer Encoder to perform Tweet Sentiment Extraction sentiment classification (three classes: positive, neutral, negative).

### Objectives:
- Load the pre-trained machine translation encoder checkpoint from Part 1.
- Fine-tune the encoder (or classify on top of it) for sentiment analysis.
- Study the effect of **removing positional embeddings** (positional embedding ablation) on classification accuracy.

## 1. Setup and Environment

In [ ]:
from pathlib import Path
import sys
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
if cwd.name == "part2_sentiment_analysis":
    PART2_DIR = cwd
    ROOT = PART2_DIR.parents[1]
else:
    ROOT = cwd
    PART2_DIR = ROOT / "code" / "part2_sentiment_analysis"

if str(PART2_DIR) not in sys.path:
    sys.path.insert(0, str(PART2_DIR))

print("ROOT:", ROOT)
print("PART2_DIR:", PART2_DIR)
print("CUDA Available:", torch.cuda.is_available())

## 2. Load Tweet Sentiment Dataset

We use the direct loader and dataset classes implemented in `train_sentiment.py` to prevent redundant code.

In [ ]:
import argparse
from train_sentiment import load_sentiment_rows, maybe_limit, build_label_mapping, TweetSentimentDataset, set_seed
from torch.utils.data import DataLoader

# Build dummy arguments to match train_sentiment expectations
args = argparse.Namespace(
    dataset_name="mteb/tweet_sentiment_extraction",
    train_file=None,
    dev_file=None,
    test_file=None,
    text_column=None,
    label_column=None,
    seed=42,
    max_train_samples=2000, # Sub-sample to keep runs fast and highly interactive
    max_dev_samples=400,
    max_test_samples=400
)

set_seed(args.seed)
train_rows, dev_rows, test_rows, text_column, label_column, data_source = load_sentiment_rows(args)
train_rows = maybe_limit(train_rows, args.max_train_samples)
dev_rows = maybe_limit(dev_rows, args.max_dev_samples)
test_rows = maybe_limit(test_rows, args.max_test_samples)

print("Data source:", data_source)
print(f"Loaded rows: Train={len(train_rows)}, Dev={len(dev_rows)}, Test={len(test_rows)}")
print(f"Text column: '{text_column}', Label column: '{label_column}'")

# Peek at a training example
import pprint
pprint.pprint(train_rows[0])

## 3. Pre-trained Model Loading & Training Helper

We fetch the best Transformer model saved during Part 1 training to serve as our base encoder checkpoint.

In [ ]:
from sentiment_model import build_sentiment_model_from_checkpoint
from train_sentiment import train_one_epoch, evaluate, default_checkpoint_path

# Locate translation checkpoint from baseline default 6 layer
checkpoint_path = ROOT / "outputs" / "part1_machine_translation" / "baseline_default_6_layer" / "best_model.pt"
if not checkpoint_path.exists():
    checkpoint_path = default_checkpoint_path()

print("Translation encoder source checkpoint:", checkpoint_path)
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)

def run_sentiment_experiment(name, use_position_embedding, epochs=5, batch_size=64, lr=2e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    output_dir = ROOT / "outputs" / "part2_sentiment_analysis" / name
    output_dir.mkdir(parents=True, exist_ok=True)
    
    set_seed(42)
    label_to_id = build_label_mapping(train_rows, dev_rows, test_rows, label_column=label_column)
    
    # Instantiate model using the checkpoint
    model = build_sentiment_model_from_checkpoint(
        checkpoint,
        num_classes=len(label_to_id),
        seq_len=60,
        dropout=0.1,
        pooling="mean",
        use_position_embedding=use_position_embedding,
    )
    model.to(device)
    
    # Prepare Datasets & Dataloaders
    train_dataset = TweetSentimentDataset(train_rows, checkpoint["en_word_dict"], label_to_id, 60, text_column, label_column)
    dev_dataset = TweetSentimentDataset(dev_rows, checkpoint["en_word_dict"], label_to_id, 60, text_column, label_column)
    test_dataset = TweetSentimentDataset(test_rows, checkpoint["en_word_dict"], label_to_id, 60, text_column, label_column)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    dev_loader = DataLoader(dev_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    loss_fn = nn.CrossEntropyLoss()
    
    metrics = []
    best_dev_acc = -1.0
    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, loss_fn, device, max_grad_norm=1.0, show_progress=False
        )
        dev_loss, dev_acc, _, _, _ = evaluate(model, dev_loader, loss_fn, device)
        metrics.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "dev_loss": dev_loss,
            "dev_accuracy": dev_acc,
        })
        print(f"[{name}] Epoch {epoch}/{epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Dev Loss: {dev_loss:.4f}, Dev Acc: {dev_acc:.4f}")
        
        if dev_acc > best_dev_acc:
            best_dev_acc = dev_acc
            torch.save(model.state_dict(), output_dir / "best_model.pt")
            
    # Load best checkpoint and evaluate on Test Set
    model.load_state_dict(torch.load(output_dir / "best_model.pt"))
    test_loss, test_acc, y_true, y_pred, test_rows_out = evaluate(model, test_loader, loss_fn, device)
    print(f"-> Test evaluation accuracy for {name}: {test_acc:.4f}")
    return metrics, test_acc, test_rows_out

## 4. Run Comparative Experiments (PE vs No PE)

In [ ]:
# Run 1: With Positional Embeddings
print("=== Fine-Tuning Sentiment Classifier WITH Positional Embeddings ===")
metrics_with, acc_with, test_with = run_sentiment_experiment(
    "with_position_embedding", use_position_embedding=True, epochs=8
)

# Run 2: Without Positional Embeddings (Ablation study)
print("\n=== Fine-Tuning Sentiment Classifier WITHOUT Positional Embeddings ===")
metrics_without, acc_without, test_without = run_sentiment_experiment(
    "without_position_embedding", use_position_embedding=False, epochs=8
)

## 5. Visualizing the Impact of Positional Embeddings

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(metrics_with) + 1)

# 1. Plot Loss curves
axes[0].plot(epochs, [m["train_loss"] for m in metrics_with], 'o-', color='tab:blue', label="With PE (Train)")
axes[0].plot(epochs, [m["dev_loss"] for m in metrics_with], 'o--', color='tab:blue', alpha=0.7, label="With PE (Dev)")
axes[0].plot(epochs, [m["train_loss"] for m in metrics_without], 's-', color='tab:orange', label="Without PE (Train)")
axes[0].plot(epochs, [m["dev_loss"] for m in metrics_without], 's--', color='tab:orange', alpha=0.7, label="Without PE (Dev)")
axes[0].set_title("Training and Dev Loss Comparison")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross Entropy Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Plot Accuracy curves
axes[1].plot(epochs, [m["train_accuracy"] for m in metrics_with], 'o-', color='tab:blue', label="With PE (Train)")
axes[1].plot(epochs, [m["dev_accuracy"] for m in metrics_with], 'o--', color='tab:blue', alpha=0.7, label="With PE (Dev)")
axes[1].plot(epochs, [m["train_accuracy"] for m in metrics_without], 's-', color='tab:orange', label="Without PE (Train)")
axes[1].plot(epochs, [m["dev_accuracy"] for m in metrics_without], 's--', color='tab:orange', alpha=0.7, label="Without PE (Dev)")
axes[1].set_title("Training and Dev Accuracy Comparison")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(ROOT / "outputs" / "part2_sentiment_analysis" / "pos_ablation_curves.png", dpi=200)
plt.show()

## 6. Results and Comparative Analysis

In [ ]:
results_df = pd.DataFrame([
    {"Experiment": "With Positional Embeddings (Sinusoidal)", "Test Accuracy": acc_with},
    {"Experiment": "Without Positional Embeddings (Ablated)", "Test Accuracy": acc_without}
])

print("=== Test Set Final Summary ===")
display(results_df)

# Load label mapping to convert IDs back to words
label_to_id = build_label_mapping(train_rows, dev_rows, test_rows, label_column=label_column)
id_to_label = {idx: label for label, idx in label_to_id.items()}

# Display sample predictions from the With PE model
samples_df = pd.DataFrame(test_with[:15])
samples_df["predicted_label"] = samples_df["pred_id"].map(id_to_label)
print("\n=== Random Sample Sentiment Predictions (With PE Model) ===")
display(samples_df[["text", "gold_label", "predicted_label"]])